# Validated GenAI Analysis of Public Comments on the FTC's Personalized-Pricing Policy

## 1. Business understanding

**Context.** In August 2026 the Federal Trade Commission (FTC) proposed an enforcement policy on *personalized pricing*: businesses using a consumer's personal data to estimate what that person will pay, and setting their price accordingly. The proposal does not ban the practice. It says businesses may break the law if they fail to clearly disclose that a price is personalized, why, and what data was used. The public comment period (docket FTC-2026-1057) closed on September 25, 2026, with 3,706 public comments.

**The business problem.** Regulators, law firms and public-affairs teams must read and summarize thousands of comments like these by hand. GenAI could do this faster and more cheaply, but a misread comment misrepresents the public record. Before anyone relies on it, the AI's accuracy has to be measured against careful human judgment.

**Research questions.**

*Workflow (for organizations adopting GenAI):*
1. How accurately does an LLM classify comments compared with human coders?
2. What does it cost per comment compared with human analysts?
3. Where does it fail?
4. What controls make it safe to use?

*Findings (for companies that use personalized pricing):*

5. What do individual commenters object to: the price difference, the use of their data, or the lack of disclosure?
6. What do they want the FTC to do?

**Scope.** This study analyzes comments from **individuals**. Submissions from organizations (companies, trade groups, advocacy groups) are mostly long attached documents that need a different method; comparing them with individuals is left for future work.

**Approach.** Understand → Use → Evaluate → Govern. Findings are reported only for labels the AI has been shown to classify reliably.

## Setup

In [ ]:
# Loads the libraries and defines the settings and file paths
# used in the rest of the notebook. Every setting that affects
# results is defined here, so it only needs changing in one place.

# Standard library
import os                      # environment variables (e.g., the API key)
import re                      # regular expressions, for cleaning text
import html                    # decodes HTML codes such as &rsquo; into normal characters
import json                    # reads and writes the LLM's structured (JSON) output
from pathlib import Path       # builds file paths that work on Windows and Mac

# Data analysis and plotting
import numpy as np             # numerical operations
import pandas as pd            # loading and manipulating the comment data
import matplotlib.pyplot as plt  # charts
import seaborn as sns          # nicer charts

# Reproducibility settings
# The same random seed was used to draw the comment sample from the API,
# so anyone rerunning the notebook gets exactly the same results.
SEED = 20261027

# Total public comments on docket FTC-2026-1057, as reported by the
# Regulations.gov API. Used to describe how much of the docket we analyze.
DOCKET_TOTAL = 3706

# File paths
# The notebook lives in Notebooks/, but the data is at the top of the repo.
# Find the repo's top folder by walking up until we find the Data folder,
# so the paths work no matter where the notebook is run from.
ROOT = Path.cwd()
while not (ROOT / "Data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Input: individual comments pulled from the Regulations.gov API.
DATA_FILE = ROOT / "Data" / "individuals_sentiment_data.csv"

# Output: folder for saved charts, at the top of the repo; created if needed.
FIG = ROOT / "outputs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

# Display settings
# Show up to 120 characters of each comment when printing tables.
pd.set_option("display.max_colwidth", 120)

# Check the input file exists before going further
# Stops the notebook with a clear message if the file is missing or misnamed,
# instead of failing later with a confusing error.
assert DATA_FILE.exists(), f"Missing {DATA_FILE}: check the folder and file name"
print("Setup complete. Data file found:", DATA_FILE)

Setup complete. Data file found: c:\Users\ADMIN\Documents\Daniels College of Business\Fall 2026\Analytics & AI\ftc-pricing-comments-genai\Data\individuals_sentiment_data.csv


## 2. Data understanding

**Source.** Public comments on docket FTC-2026-1057, retrieved from the official Regulations.gov v4 API. All comment IDs on the docket were shuffled with a fixed seed (20261027) and fetched in that order, so the comments retrieved form a simple random sample of the docket.

**Unit of observation.** One public comment submitted by an individual.

**Privacy.** Commenters' names were excluded at download. Only the comment text, posting date and, where given, the U.S. state are kept.